In [2]:
clinical_note = """
Monsieur Jean Dupont, âgé de 67 ans, est admis le 12 mars 2023 aux urgences pour dyspnée aiguë et douleur thoracique. 
Le patient présente des antécédents d’hypertension artérielle et de diabète de type 2.

À l’examen clinique, on note une saturation à 88%, une pression artérielle à 170/95 mmHg et une glycémie à 1.82 g/L.

Un scanner thoracique confirme une embolie pulmonaire bilatérale.

Traitement initié : Héparine 5000 UI en bolus IV puis relais par Apixaban 5 mg deux fois par jour pour une durée prévue de 6 mois.

Le patient est transféré en cardiologie le 14 mars 2023.
"""

In [16]:
from spacy.cli import download
download("fr_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 10.4 MB/s eta 0:00:00m eta 0:00:010:01:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import re
import spacy

nlp = spacy.load("fr_core_news_md")

def extract_clinical_information(text: str) -> dict:
    doc = nlp(text)

    result = {
        "identity": {
            "name": None,
            "age": None
        },
        "dates": {
            "admission": None,
            "transfert": None
        },
        "medical": {
            "motif_admission": None,
            "antecedents": [],
            "diagnostic": None
        },
        "treatment": {
            "medications": [],
            "dosages": [],
            "route": None,
            "duration": None
        },
        "measurements": {
            "saturation": None,
            "pression_arterielle": None,
            "glycemie": None
        }
    }

    for ent in doc.ents:
        if ent.label_ == "PER":
            result["identity"]["name"] = ent.text
        if ent.label_ == "DATE":
            # Dates classifiées par contexte
            sentence = ent.sent.text.lower()
            if "admis" in sentence:
                result["dates"]["admission"] = ent.text
            if "transféré" in sentence:
                result["dates"]["transfert"] = ent.text

    age_match = re.search(r"\b(\d+)\s+ans\b", text.lower())
    if age_match:
        result["identity"]["age"] = int(age_match.group(1))

    for token in doc:
        if token.lemma_ == "admettre":
            for child in token.children:
                if child.text.lower() == "pour":
                    subtree = list(child.subtree)
                    motif = " ".join([t.text for t in subtree])
                    result["medical"]["motif_admission"] = motif.replace("pour ", "").strip()

    for sent in doc.sents:
        if "antécédent" in sent.text.lower():
            for token in sent:
                if token.pos_ == "NOUN":
                    result["medical"]["antecedents"].append(token.text)

    for sent in doc.sents:
        if "confirme" in sent.text.lower():
            for token in sent:
                if token.dep_ == "obj":
                    result["medical"]["diagnostic"] = token.text

    treatment_section = False
    for sent in doc.sents:
        if "traitement initié" in sent.text.lower():
            treatment_section = True

        if treatment_section:
            # Médicaments = PROPN ou tokens majuscule init
            for token in sent:
                if token.pos_ == "PROPN":
                    result["treatment"]["medications"].append(token.text)

            # Dosages
            dosage_matches = re.findall(r"\b\d+\s?(mg|ui|g)\b", sent.text.lower())
            if dosage_matches:
                full_dosages = re.findall(r"\b\d+\s?(?:mg|ui|g)\b", sent.text.lower())
                result["treatment"]["dosages"].extend(full_dosages)

            # Voie
            if "iv" in sent.text.lower():
                result["treatment"]["route"] = "IV"

            # Durée
            duration_match = re.search(r"\b\d+\s+(jours?|mois)\b", sent.text.lower())
            if duration_match:
                result["treatment"]["duration"] = duration_match.group(0)

    saturation_match = re.search(r"saturation.*?(\d+)%", text.lower())
    if saturation_match:
        result["measurements"]["saturation"] = int(saturation_match.group(1))

    tension_match = re.search(r"(\d+/\d+)\s*mmhg", text.lower())
    if tension_match:
        result["measurements"]["pression_arterielle"] = tension_match.group(1)

    glycemie_match = re.search(r"(\d+(?:\.\d+)?)\s*g/l", text.lower())
    if glycemie_match:
        result["measurements"]["glycemie"] = {
            "value": float(glycemie_match.group(1)),
            "unit": "g/L"
        }

    return result


# Utilisation
clinical_data = extract_clinical_information(clinical_note)
print(clinical_data)

{'identity': {'name': 'Jean Dupont', 'age': 67}, 'dates': {'admission': None, 'transfert': None}, 'medical': {'motif_admission': None, 'antecedents': ['patient', 'antécédents', 'hypertension', 'diabète', 'type', 'examen', 'saturation', '%'], 'diagnostic': 'embolie'}, 'treatment': {'medications': ['Apixaban'], 'dosages': ['5000 ui', '5 mg'], 'route': 'IV', 'duration': '6 mois'}, 'measurements': {'saturation': 88, 'pression_arterielle': '170/95', 'glycemie': {'value': 1.82, 'unit': 'g/L'}}}


In [5]:
doc = nlp(clinical_note)

In [6]:
doc.ents

(Jean Dupont, Héparine 5000 UI, IV, Apixaban 5 mg)

In [7]:
for ent in doc.ents:
    print(ent.text, ent.label_)

Jean Dupont PER
Héparine 5000 UI MISC
IV LOC
Apixaban 5 mg MISC


In [11]:
name = None
age = None

for ent in doc.ents:
    if ent.label_ == "PER":
        name = ent.text

age_match = re.search(r"\b(\d+)\s+ans\b", clinical_note.lower())
if age_match:
    age = int(age_match.group(1))

print("Name:", name)
print("Age:", age)

Name: Jean Dupont
Age: 67


In [13]:
admission_date = None
transfert_date = None

date_pattern = r"\b\d{1,2}\s+\w+\s+\d{4}\b"

for sent in doc.sents:
    sent_text = sent.text.lower()

    date_match = re.search(date_pattern, sent_text)
    if date_match:
        date_value = date_match.group(0)

        # On utilise spaCy pour comprendre le contexte
        for token in sent:
            if token.lemma_ == "admettre":
                admission_date = date_value
            if token.lemma_ == "transférer":
                transfert_date = date_value

print("Admission:", admission_date)
print("Transfert:", transfert_date)

Admission: 12 mars 2023
Transfert: 14 mars 2023


In [15]:
motif = None

for token in doc:
    if token.lemma_ == "admettre":
        for child in token.children:
            if child.text.lower() == "pour":
                subtree = list(child.subtree)
                motif = " ".join([t.text for t in subtree])
                motif = motif.replace("pour ", "").strip()

print("Motif:", motif)

Motif: None


In [16]:
antecedents = []

for sent in doc.sents:
    if "antécédent" in sent.text.lower():
        for token in sent:
            if token.pos_ == "NOUN":
                antecedents.append(token.text)

print("Antécédents:", antecedents)

Antécédents: ['patient', 'antécédents', 'hypertension', 'diabète', 'type', 'examen', 'saturation', '%']


In [17]:
diagnostic = None

for sent in doc.sents:
    if "confirme" in sent.text.lower():
        for token in sent:
            if token.dep_ == "obj":
                diagnostic = token.text

print("Diagnostic:", diagnostic)

Diagnostic: embolie


In [18]:
medications = []
dosages = []
route = None
duration = None

for sent in doc.sents:
    if "traitement initié" in sent.text.lower():

        for token in sent:
            if token.pos_ == "PROPN":
                medications.append(token.text)

        dosage_matches = re.findall(r"\b\d+\s?(?:mg|ui|g)\b", sent.text.lower())
        dosages.extend(dosage_matches)

        if "iv" in sent.text.lower():
            route = "IV"

        duration_match = re.search(r"\b\d+\s+(?:jours?|mois)\b", sent.text.lower())
        if duration_match:
            duration = duration_match.group(0)

print("Medications:", medications)
print("Dosages:", dosages)
print("Route:", route)
print("Duration:", duration)

Medications: ['Apixaban']
Dosages: ['5000 ui', '5 mg']
Route: IV
Duration: 6 mois
